# 数据统计

In [1]:
import pandas as pd
import json
import networkx as nx
import matplotlib.pyplot as plt
import math

from collections import defaultdict

In [2]:
import sys
sys.path.append('..')

## 原始数据

In [3]:
words_file_path = 'data_output/liuyuning/qq_words_data.csv'
df_words_raw = pd.read_csv(words_file_path)
df_words_raw

,song_id,word,pos,freq,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,album_name,album_id,album_mid,duration,publish_time,tv_name,song_name_unique,publish_date,is_ost
0,633973719,处,n,4,0029blSZ16tCiJ,纵马踏歌行,《剑来》动画第二季插曲,刘宇宁,2241311,001Iu4Dv1NzRCD,83047554,纵马踏歌行,003otjHh1KR2Lg,176,1768320000,剑来,纵马踏歌行,2026-01-14,1
1,633973719,了,ul,4,0029blSZ16tCiJ,纵马踏歌行,《剑来》动画第二季插曲,刘宇宁,2241311,001Iu4Dv1NzRCD,83047554,纵马踏歌行,003otjHh1KR2Lg,176,1768320000,剑来,纵马踏歌行,2026-01-14,1
2,633973719,把,p,2,0029blSZ16tCiJ,纵马踏歌行,《剑来》动画第二季插曲,刘宇宁,2241311,001Iu4Dv1NzRCD,83047554,纵马踏歌行,003otjHh1KR2Lg,176,1768320000,剑来,纵马踏歌行,2026-01-14,1
3,633973719,江湖,ns,2,0029blSZ16tCiJ,纵马踏歌行,《剑来》动画第二季插曲,刘宇宁,2241311,001Iu4Dv1NzRCD,83047554,纵马踏歌行,003otjHh1KR2Lg,176,1768320000,剑来,纵马踏歌行,2026-01-14,1
4,633973719,斟入,v,2,0029blSZ16tCiJ,纵马踏歌行,《剑来》动画第二季插曲,刘宇宁,2241311,001Iu4Dv1NzRCD,83047554,纵马踏歌行,003otjHh1KR2Lg,176,1768320000,剑来,纵马踏歌行,2026-01-14,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13880,297559879,血泪,n,1,000sfLHg43UbQo,苍穹之下,电视剧《斗罗大陆》戴沐白人物曲）,刘宇宁,2241311,001Iu4Dv1NzRCD,17500078,斗罗大陆 史兰客七怪音乐专辑,000uo5Ig1624LG,272,1613354400,斗罗大陆,苍穹之下,2021-02-15,1
13881,297559879,挥洒,v,1,000sfLHg43UbQo,苍穹之下,电视剧《斗罗大陆》戴沐白人物曲）,刘宇宁,2241311,001Iu4Dv1NzRCD,17500078,斗罗大陆 史兰客七怪音乐专辑,000uo5Ig1624LG,272,1613354400,斗罗大陆,苍穹之下,2021-02-15,1
13882,297559879,书写,n,1,000sfLHg43UbQo,苍穹之下,电视剧《斗罗大陆》戴沐白人物曲）,刘宇宁,2241311,001Iu4Dv1NzRCD,17500078,斗罗大陆 史兰客七怪音乐专辑,000uo5Ig1624LG,272,1613354400,斗罗大陆,苍穹之下,2021-02-15,1
13883,297559879,新,a,1,000sfLHg43UbQo,苍穹之下,电视剧《斗罗大陆》戴沐白人物曲）,刘宇宁,2241311,001Iu4Dv1NzRCD,17500078,斗罗大陆 史兰客七怪音乐专辑,000uo5Ig1624LG,272,1613354400,斗罗大陆,苍穹之下,2021-02-15,1


## 数据处理

In [4]:
def clear(df):
    # 曲目，专辑添加唯一id
    df = df.copy()
    df['song_id_unique'] = 'song' + df['song_id'].astype(str)
    # df_album_fixed = df[['album_id', 'album_fixed']].drop_duplicates(subset=['album_fixed'], keep='first').reset_index(drop=True)
    # df_album_fixed['album_id_unique'] = 'album' + df_album_fixed['album_id'].astype(int).astype(str)
    # df_album_fixed = df_album_fixed[['album_id_unique', 'album_fixed']]
    df['song_year'] = df['publish_date'].str.split('-').str[0]
    df['album_fixed'] = df['song_year'] + '年'
    df['album_id_unique'] = 'album' + df['song_year']
    df['legend_type'] = df['song_year'] + '年'

    # df['legend_order'] = df['legend_type']
    # 词唯一id
    df_words = df[['word', 'pos']].drop_duplicates().reset_index(drop=False)
    df_words['word_id'] = 'word' + df_words['index'].astype(str)
    df_words = df_words.drop('index', axis=1).reset_index(drop=True)
    df = df.merge(df_words, on=['word', 'pos'], how='left')
    return df

df_words = clear(df_words_raw)
df_words

,song_id,word,pos,freq,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,...,tv_name,song_name_unique,publish_date,is_ost,song_id_unique,song_year,album_fixed,album_id_unique,legend_type,word_id
0,633973719,处,n,4,0029blSZ16tCiJ,纵马踏歌行,《剑来》动画第二季插曲,刘宇宁,2241311,001Iu4Dv1NzRCD,...,剑来,纵马踏歌行,2026-01-14,1,song633973719,2026,2026年,album2026,2026年,word0
1,633973719,了,ul,4,0029blSZ16tCiJ,纵马踏歌行,《剑来》动画第二季插曲,刘宇宁,2241311,001Iu4Dv1NzRCD,...,剑来,纵马踏歌行,2026-01-14,1,song633973719,2026,2026年,album2026,2026年,word1
2,633973719,把,p,2,0029blSZ16tCiJ,纵马踏歌行,《剑来》动画第二季插曲,刘宇宁,2241311,001Iu4Dv1NzRCD,...,剑来,纵马踏歌行,2026-01-14,1,song633973719,2026,2026年,album2026,2026年,word2
3,633973719,江湖,ns,2,0029blSZ16tCiJ,纵马踏歌行,《剑来》动画第二季插曲,刘宇宁,2241311,001Iu4Dv1NzRCD,...,剑来,纵马踏歌行,2026-01-14,1,song633973719,2026,2026年,album2026,2026年,word3
4,633973719,斟入,v,2,0029blSZ16tCiJ,纵马踏歌行,《剑来》动画第二季插曲,刘宇宁,2241311,001Iu4Dv1NzRCD,...,剑来,纵马踏歌行,2026-01-14,1,song633973719,2026,2026年,album2026,2026年,word4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13880,297559879,血泪,n,1,000sfLHg43UbQo,苍穹之下,电视剧《斗罗大陆》戴沐白人物曲）,刘宇宁,2241311,001Iu4Dv1NzRCD,...,斗罗大陆,苍穹之下,2021-02-15,1,song297559879,2021,2021年,album2021,2021年,word1214
13881,297559879,挥洒,v,1,000sfLHg43UbQo,苍穹之下,电视剧《斗罗大陆》戴沐白人物曲）,刘宇宁,2241311,001Iu4Dv1NzRCD,...,斗罗大陆,苍穹之下,2021-02-15,1,song297559879,2021,2021年,album2021,2021年,word13881
13882,297559879,书写,n,1,000sfLHg43UbQo,苍穹之下,电视剧《斗罗大陆》戴沐白人物曲）,刘宇宁,2241311,001Iu4Dv1NzRCD,...,斗罗大陆,苍穹之下,2021-02-15,1,song297559879,2021,2021年,album2021,2021年,word12097
13883,297559879,新,a,1,000sfLHg43UbQo,苍穹之下,电视剧《斗罗大陆》戴沐白人物曲）,刘宇宁,2241311,001Iu4Dv1NzRCD,...,斗罗大陆,苍穹之下,2021-02-15,1,song297559879,2021,2021年,album2021,2021年,word10044


# 分词数据
名词： n, w
动词： v
形容词： v

In [5]:
def word_count_by_pos(df, pos, words_num=30, is_starts_with=True):
    if is_starts_with:
        df_sub = df[df['pos'].str.startswith(
            pos, na=False)]
        # 歌曲数
        word_cnt = df_sub.groupby('word')['song_id'].nunique().reset_index().rename(columns={'song_id': 'song_num'})
        word_sum = df[df['pos'].str.startswith(
            pos, na=False)].groupby('word')['freq'].sum().reset_index()
    else:
        word_cnt = df[df['pos']==pos]['word'].groupby('word')['song_id'].nunique().reset_index().rename(columns={'song_id': 'song_num'})
        word_sum = df[df['pos']==pos].groupby('word')['freq'].sum().reset_index()
    if words_num:
        word_cnt = word_cnt.sort_values(by='song_num', ascending=False)
        res = word_cnt.head(words_num).merge(word_sum, on='word', how='left')
    else:
        res = word_cnt.merge(word_sum, on='word', how='left')
    res['order'] = 100 - res.index
    # res = res.rename(columns={
    #     'count': 'songs_num',
    # })
    return res

# 词图

## 二分图布局

In [6]:
# 120度弧线布局
def generate_symmetric_bipartite_layout(nodes):
    coords = {}
    
    # --- 1. 参数定义 ---
    # 定义两条对称弧线的几何参数
    arc_radius = 800
    arc_span = math.pi / 1.5  # 约 120 度
    # 计算弧线的垂直跨度 (用于决定 Word 长度)
    arc_vertical_span = 2 * arc_radius * math.sin(arc_span / 2)
    
    # --- 2. Word 节点 (画布中央直线) ---
    word_nodes = sorted([n for n in nodes if n['type'] == 'word'], key=lambda x: x['degree'], reverse=True)
    num_words = len(word_nodes)
    
    # 长度为弧线跨度的 90%
    target_word_height = arc_vertical_span * 0.9
    word_y_gap = target_word_height / (num_words - 1) if num_words > 1 else 0

    for i, node in enumerate(word_nodes):
        # 中心向两端扩散逻辑: 0->0, 1->1, 2->-1, 3->2...
        rank = (i + 1) // 2
        direction = 1 if i % 2 != 0 else -1
        if i == 0: direction = 0
        
        # Word 位于 x=0，且在 y 轴居中
        coords[node['id']] = (0, round(rank * direction * word_y_gap, 2))

    # --- 3. Song 节点 (两侧对称弧线) ---
    song_nodes = [n for n in nodes if n['type'] == 'song']
    mid_idx = len(song_nodes) // 2
    # 将 Song 平均分为两组：左侧弧和右侧弧
    song_columns = [song_nodes[:mid_idx], song_nodes[mid_idx:]]
    
    # 弧线布局配置：[左侧弧, 右侧弧]
    # 左侧弧圆心在正 X，向左弯曲；右侧弧圆心在负 X，向右弯曲
    configs = [
        {"center_x": 0, "direction": -1}, # 右侧弧 (位于 Word 右侧)
        {"center_x": 0, "direction": 1}  # 左侧弧 (位于 Word 左侧)
    ]

    for col_idx, col_items in enumerate(song_columns):
        config = configs[col_idx]
        num_in_col = len(col_items)
        if num_in_col == 0: continue
        
        for row_idx, node in enumerate(col_items):
            # 从上到下均匀分布角度
            if num_in_col > 1:
                angle = (arc_span / 2) - (row_idx / (num_in_col - 1)) * arc_span
            else:
                angle = 0
            
            # 计算坐标
            # cos(angle) 决定 X 偏移，direction 决定是在圆心左侧还是右侧
            x = config["center_x"] + config["direction"] * (arc_radius * math.cos(angle))
            y = arc_radius * math.sin(angle)
            coords[node['id']] = (round(x, 2), round(y, 2))
            
    return coords


In [7]:
# 112度弧形布局
def generate_embracing_layout(nodes):
    coords = {}
    
    # --- 1. 几何参数设定 ---
    arc_radius = 800           # 半径
    arc_span = math.pi / 1.6   # 弧度张角 (约112度)
    # 弧开口端点距离中心直线的水平间距
    horizontal_gap = 300       
    
    # 计算弧线端点的 Y 轴跨度 (用于对齐 Word)
    # y = r * sin(theta)
    arc_half_height = arc_radius * math.sin(arc_span / 2)
    arc_total_height = 2 * arc_half_height
    
    # --- 2. Word 节点 (居中直线) ---
    word_nodes = sorted([n for n in nodes if n['type'] == 'word'], key=lambda x: x['degree'], reverse=True)
    num_words = len(word_nodes)
    
    # 长度为弧垂直跨度的 90%
    target_word_height = arc_total_height * 0.9
    word_y_gap = target_word_height / (num_words - 1) if num_words > 1 else 0

    for i, node in enumerate(word_nodes):
        # 中心扩散排序
        rank = (i + 1) // 2
        direction = 1 if i % 2 != 0 else -1
        if i == 0: direction = 0
        coords[node['id']] = (0, round(rank * direction * word_y_gap, 2))

    # --- 3. Song 节点 (开口向内的对称弧线) ---
    song_nodes = [n for n in nodes if n['type'] == 'song']
    mid_idx = len(song_nodes) // 2
    song_groups = [song_nodes[:mid_idx], song_nodes[mid_idx:]]
    
    # 配置说明：
    # 为了让开口面向直线 (x=0)：
    # 左侧弧的圆心要在右侧，x 坐标为 (horizontal_gap + radius * cos(half_span))
    # 右侧弧的圆心要在左侧，x 坐标为 -(horizontal_gap + radius * cos(half_span))
    
    # 计算圆心位置，使得弧的端点正好落在 horizontal_gap 线上
    edge_x_offset = arc_radius * math.cos(arc_span / 2)
    center_x_pos = horizontal_gap + edge_x_offset

    side_configs = [
        {"c_x": 0, "dir": 1, "name": "Right"}, # 右侧弧，圆心在左，向右弯
        {"c_x": 0, "dir": -1, "name": "Left"}   # 左侧弧，圆心在右，向左弯
    ]

    for col_idx, col_items in enumerate(song_groups):
        conf = side_configs[col_idx]
        num_in_col = len(col_items)
        
        for row_idx, node in enumerate(col_items):
            # 角度分布
            angle = (arc_span / 2) - (row_idx / (num_in_col - 1)) * arc_span if num_in_col > 1 else 0
            
            # 计算 X: 圆心 + 方向 * (半径 * cos(角度))
            x = conf["c_x"] + conf["dir"] * (arc_radius * math.cos(angle))
            y = arc_radius * math.sin(angle)
            coords[node['id']] = (round(x, 2), round(y, 2))
            
    return coords


## 数据处理

In [8]:
def get_subset_data(df_raw, pos, words_num, is_starts_with):
    # 1. 获取符合特定词性的高频词集合
    # 假设 word_count_by_pos 返回的是一个包含 'word' 列的 DataFrame
    words_set = word_count_by_pos(df_raw, pos=pos, words_num=words_num, is_starts_with=is_starts_with)
    target_words = set(words_set['word']) # 转为 set 匹配速度更快

    # 2. 统一词性过滤逻辑
    if is_starts_with:
        mask = df_raw['pos'].str.startswith(pos, na=False)
    else:
        mask = df_raw['pos'] == pos
    
    # 3. 筛选、清洗并保留必要的列
    # 链式操作：过滤词性 -> 过滤高频词 -> 执行自定义清洗
    df_subset = df_raw[mask].copy()
    df_subset = df_subset[df_subset['word'].isin(target_words)]
    
    # 4. 统一词性标签（既然是 Subset，统一设为传入的 pos）
    df_subset['pos'] = pos
    # 统一词的id
    df_subset = clear(df_subset)

    # 5. 生成年份排序映射（album_order）
    # 使用 factorize 可以直接一步生成按顺序排列的编码
    # 如果必须按年份数值排序，则先排序再 factorize
    unique_years = sorted(df_subset['song_year'].unique())
    year_to_order = {year: i for i, year in enumerate(unique_years)}
    df_subset['album_order'] = df_subset['song_year'].map(year_to_order)

    return df_subset.reset_index(drop=True)

## json数据输出

In [9]:
def get_word_subset_graph(G_words_subset, df_subset):
    word_graph_dict = defaultdict(list)
    words_degree = G_words_subset.degree
    nodes_subset = []
    for n in G_words_subset.nodes():
        n_dict = {
            'id': n,
            'degree': words_degree[n],
            'type': 'word' if 'word' in n else 'song',
        }
        nodes_subset.append(n_dict)
    pos = generate_embracing_layout(nodes_subset)

    for node in G_words_subset.nodes:
        nodes_dict = defaultdict(str)
        nodes_dict['id'] = node
        nodes_dict['size'] = words_degree[node]
        nodes_dict['x'] = pos[node][0]
        nodes_dict['y'] = pos[node][1]
        if 'word' in node:
            nodes_dict['label'] = df_subset[df_subset['word_id'] ==
                                            node]['word'].values[0]
            nodes_dict['node_type'] = 'word'
            nodes_dict['album_id'] = ""
            nodes_dict['album'] = ""
            nodes_dict['data'] = {'cluster': '词'}
        elif 'song' in node:
            nodes_dict['label'] = df_subset[df_subset['song_id_unique'] ==
                                            node]['song_name'].values[0]
            nodes_dict['node_type'] = 'song'
            nodes_dict['album_id'] = df_subset[
                df_subset['song_id_unique'] ==
                node]['album_id_unique'].values[0]
            nodes_dict['album'] = df_subset[df_subset['song_id_unique'] ==
                                            node]['album_fixed'].values[0]
            nodes_dict['album_order'] = int(df_subset[
                df_subset['song_id_unique'] == node]['album_order'].values[0])
            nodes_dict['tv_name'] = df_subset[df_subset['song_id_unique'] ==
                                              node]['tv_name'].values[0]
            nodes_dict['data'] = {
                'cluster':
                df_subset[df_subset['song_id_unique'] == node]
                ['album_fixed'].values[0]
            }
        word_graph_dict['nodes'].append(nodes_dict)
    for edge in G_words_subset.edges:
        edges_dict = defaultdict(str)
        edges_dict['source'] = edge[0]
        edges_dict['target'] = edge[1]
        for i in edge:
            if 'song' in i:
                edges_dict['album'] = df_subset[df_subset['song_id_unique'] ==
                                                i]['album_fixed'].values[0]
                edges_dict['album_order'] = int(
                    df_subset[df_subset['song_id_unique'] ==
                              i]['album_order'].values[0])
            else:
                edges_dict['album'] = ""
                edges_dict['album_order'] = ""

        word_graph_dict['edges'].append(edges_dict)
    return word_graph_dict

In [17]:
df_subset = get_subset_data(df_words_raw, 'a', words_num=50, is_starts_with=True)
df_subset

,song_id,word,pos,freq,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,...,song_name_unique,publish_date,is_ost,song_id_unique,song_year,album_fixed,album_id_unique,legend_type,word_id,album_order
0,633973719,旧,a,2,0029blSZ16tCiJ,纵马踏歌行,《剑来》动画第二季插曲,刘宇宁,2241311,001Iu4Dv1NzRCD,...,纵马踏歌行,2026-01-14,1,song633973719,2026,2026年,album2026,2026年,word17,8
1,633973719,孤,a,2,0029blSZ16tCiJ,纵马踏歌行,《剑来》动画第二季插曲,刘宇宁,2241311,001Iu4Dv1NzRCD,...,纵马踏歌行,2026-01-14,1,song633973719,2026,2026年,album2026,2026年,word30,8
2,629395245,倔强,a,2,003cEvPA3QuEqn,荣光,《风过留痕》影视剧片尾曲,刘宇宁,2241311,001Iu4Dv1NzRCD,...,荣光,2026-02-03,1,song629395245,2026,2026年,album2026,2026年,word95,8
3,629395245,平凡,a,1,003cEvPA3QuEqn,荣光,《风过留痕》影视剧片尾曲,刘宇宁,2241311,001Iu4Dv1NzRCD,...,荣光,2026-02-03,1,song629395245,2026,2026年,album2026,2026年,word123,8
4,370388317,清澈,a,1,001l38Gx3sl3kP,寻一个你,《苍兰诀》电视剧温情主题曲,刘宇宁,2241311,001Iu4Dv1NzRCD,...,寻一个你,2022-08-09,1,song370388317,2022,2022年,album2022,2022年,word200,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
322,264776262,好,a,2,002rs2Tk3fYVGh,要替我幸福,《暖暖，请多指教》电视剧片尾曲,刘宇宁,2241311,001Iu4Dv1NzRCD,...,要替我幸福,2020-05-15,1,song264776262,2020,2020年,album2020,2020年,word890,2
323,264776262,美好,a,1,002rs2Tk3fYVGh,要替我幸福,《暖暖，请多指教》电视剧片尾曲,刘宇宁,2241311,001Iu4Dv1NzRCD,...,要替我幸福,2020-05-15,1,song264776262,2020,2020年,album2020,2020年,word6823,2
324,264776262,遗憾,a,1,002rs2Tk3fYVGh,要替我幸福,《暖暖，请多指教》电视剧片尾曲,刘宇宁,2241311,001Iu4Dv1NzRCD,...,要替我幸福,2020-05-15,1,song264776262,2020,2020年,album2020,2020年,word431,2
325,264776262,自由,a,1,002rs2Tk3fYVGh,要替我幸福,《暖暖，请多指教》电视剧片尾曲,刘宇宁,2241311,001Iu4Dv1NzRCD,...,要替我幸福,2020-05-15,1,song264776262,2020,2020年,album2020,2020年,word1078,2


In [18]:
df_subset['word'].nunique()

50

In [19]:
df_subset['song_id'].nunique()

137

In [20]:
df_subset[df_subset['word'] == '时']

,song_id,word,pos,freq,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,...,song_name_unique,publish_date,is_ost,song_id_unique,song_year,album_fixed,album_id_unique,legend_type,word_id,album_order


In [21]:
G_words = nx.from_pandas_edgelist(df_words, 'word_id', 'song_id_unique', edge_attr=True, create_using=nx.Graph())
G_words_subset = nx.from_pandas_edgelist(df_subset, 'word_id', 'song_id_unique', edge_attr=True, create_using=nx.Graph())
G_songs = nx.from_pandas_edgelist(df_words, 'song_id_unique', 'album_id_unique', edge_attr=True, create_using=nx.Graph())
G_words.number_of_nodes(), G_words_subset.number_of_nodes(), G_songs.number_of_nodes()

(6074, 187, 159)

In [22]:
word_graph_dict = get_word_subset_graph(G_words_subset, df_subset)

In [23]:
words_file_path = 'data_output/liuyuning/word_graph_data.json'
with open(words_file_path, 'w', encoding='utf-8') as f:
    json.dump(word_graph_dict, f, ensure_ascii=False, indent=4)